# PoreMind 面向对象逐步骤分析
需求对应六步：降噪→事件检测→特征提取→异常过滤→建模选优→新样本分类。

In [ ]:
from poremind import create_analysis_object

In [ ]:
sample_paths = {
    'std_A_01': 'std_A_01.abf',
    'std_A_02': 'std_A_02.abf',
    'std_B_01': 'std_B_01.abf',
}
sample_to_group = {
    'std_A_01': 'A',
    'std_A_02': 'A',
    'std_B_01': 'B',
}
analysis = create_analysis_object(sample_paths, sample_to_group=sample_to_group, reader='abf').load()

In [ ]:
# Step 1: 降噪 + 指定时间范围可视化（默认 butterworth_filtfilt）
analysis.denoise()
analysis.preview_signal(next(iter(analysis.traces.keys())), start_s=0.0, end_s=0.002).head()

In [ ]:
# Step 1.1: analysis.pl.current 绘制 0-1 ms 电流片段
analysis.pl.current(current='denoise', start_ms=0.0, end_ms=1.0, width=10, height=3)

In [ ]:
# Step 2: 事件检测（可切换 threshold / zscore_threshold / cusum / pelt / hmm）
analysis.detect_events(detect_method='threshold', detect_params={'sigma_k': 5.0, 'min_duration_s': 2e-4}, baseline_method='rolling_quantile', baseline_params={'window': 501, 'q': 0.5})

In [ ]:
# Step 3: 特征提取（支持自定义特征函数）
def custom_shape_features(seg):
    import numpy as np
    return {'p2p': float(np.max(seg) - np.min(seg)), 'energy': float(np.sum(seg**2))}

feat_df = analysis.extract_features(custom_feature_fns={'shape': custom_shape_features})
feat_df.head()

In [ ]:
# Step 4: 异常事件过滤（默认blockade_gmm，基于blockade_ratio）
filtered_df = analysis.filter_events()
filtered_df[['sample_id', 'event_id', 'blockade_ratio', 'quality_tag']].head()

In [ ]:
# Step 5: 10折比较多模型并选择最优（加权指标）
best_pkg = analysis.build_best_model(cv=10, scoring='accuracy')
best_pkg['best_model'], best_pkg['scores']

In [ ]:
# Step 5.1: 指定模型的10折测试集混淆矩阵
analysis.pl.model_cm(model_name=best_pkg['best_model'], split='test')

In [ ]:
# Step 5.2: 所有模型的指标柱状图（默认accuracy）
analysis.pl.model_metric_bar(metric='accuracy', split='test')

In [ ]:
# Step 6: 复用流程 + 最优模型，对新样本逐事件分类
new_paths = {'unknown_01': 'unknown_01.abf'}
pred_df = analysis.classify_new_samples(new_paths, reader='abf')
pred_df[['sample_id', 'event_id', 'start_time_s', 'end_time_s', 'pred_label']].head()